# Data Collection and Pre-Processing Lab

This notebook demonstrates an end-to-end data engineering workflow using a synthetic e-commerce transaction dataset. The dataset contains 500 transaction records and includes customer, product, pricing, coupon, and shipping information.

## Step 1 - Hello, Data!

The raw e-commerce transaction dataset is loaded from a CSV file using Pandas. The first three records are displayed to verify that the dataset has been loaded successfully.

In [ ]:
import pandas as pd
import src.transaction
Transaction = src.transaction.Transaction

df = pd.read_csv("data/ecommerce_transactions.csv")

df.head(3)

,transaction_id,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,1,2026-06-10,CUST5572,Smartwatch,367.57,2,NaN,London
1,2,2026-06-10,CUST8096,Laptop,433.53,1,SAVE10,Waterloo
2,3,2026-09-14,CUST6154,Tablet,1001.78,2,SAVE10,London


## Step 2 - Pick the Right Container

**Dictionary:** A dictionary is used to represent a transaction because it stores values with meaningful field names such as `product`, `price`, and `quantity`. It also allows values to be updated during data cleaning.

**Namedtuple:** A namedtuple was not selected because its values cannot be changed after it is created. Since the transaction data needs to be cleaned and updated, a dictionary is more suitable.

**Set:** A set is used to store unique values without duplicates. In this project, it is used to identify the unique shipping cities in the transaction data.

In [3]:
sample_transaction = df.iloc[0].to_dict()

sample_transaction

{'transaction_id': 1,
 'date': '2026-06-10',
 'customer_id': 'CUST5572',
 'product': 'Smartwatch',
 'price': 367.57,
 'quantity': 2,
 'coupon_code': nan,
 'shipping_city': 'London'}

## Step 3 - Implement Functions and Data Structure

A `Transaction` class is used to represent an individual e-commerce transaction. The class stores the transaction attributes and provides reusable methods for cleaning text values and calculating the transaction total. This keeps transaction-related processing organized and reusable.

In [4]:
from src.transaction import Transaction

row = df.iloc[0]

transaction = Transaction(
    transaction_id=row["transaction_id"],
    date=row["date"],
    customer_id=row["customer_id"],
    product=row["product"],
    price=row["price"],
    quantity=row["quantity"],
    coupon_code=row["coupon_code"],
    shipping_city=row["shipping_city"]
)

transaction.clean(df["price"].mean())

print("Product:", transaction.product)
print("Shipping City:", transaction.shipping_city)
print("Transaction Total:", transaction.total())

Product: Smartwatch
Shipping City: London
Transaction Total: 735.14


## Step 4 - Bulk Loaded

The Pandas DataFrame is converted into a list of dictionaries, where each dictionary represents one transaction. This structure provides named access to each field while allowing the complete collection of transactions to be processed using standard Python operations.

In [5]:
transactions_dict = df.to_dict(orient="records")

print("Number of transactions:", len(transactions_dict))
print("\nFirst transaction:")
print(transactions_dict[0])
print("Container type:", type(transactions_dict))
print("Individual record type:", type(transactions_dict[0]))

Number of transactions: 500

First transaction:
{'transaction_id': 1, 'date': '2026-06-10', 'customer_id': 'CUST5572', 'product': 'Smartwatch', 'price': 367.57, 'quantity': 2, 'coupon_code': nan, 'shipping_city': 'London'}
Container type: <class 'list'>
Individual record type: <class 'dict'>


## Step 5 - Quick Profiling

Quick profiling is performed to understand the basic characteristics of the transaction data before cleaning. The minimum, mean, and maximum product prices are calculated, and a Python set is used to identify unique shipping cities. Using a set is appropriate because it automatically stores only unique values.

Price statistics

In [6]:
minimum_price = df["price"].min()
mean_price = df["price"].mean()
maximum_price = df["price"].max()

print(f"Minimum Price: ${minimum_price:.2f}")
print(f"Mean Price: ${mean_price:.2f}")
print(f"Maximum Price: ${maximum_price:.2f}")

Minimum Price: $21.44
Mean Price: $1020.34
Maximum Price: $1994.12


Unique cities using a set

In [7]:
unique_cities = set(df["shipping_city"].dropna())

print("Unique shipping cities:")
print(unique_cities)

print("\nNumber of unique cities:", len(unique_cities))

Unique shipping cities:
{'Toronto', ' toronto ', 'London', 'kitchener', 'Kitchener', 'WATERLOO', 'Ottawa', 'Waterloo'}

Number of unique cities: 8


## Step 6 - Spot the Grime

The dataset is checked for problems before cleaning. We found missing prices, missing product names, and different formats of shipping city names. We also checked for missing coupon codes because some transactions may not have used a coupon.

Check missing values

In [8]:
missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values)

Missing values:
transaction_id      0
date                0
customer_id         0
product             2
price               3
quantity            0
coupon_code       130
shipping_city       0
dtype: int64


Inspect city inconsistencies

In [9]:
print("Shipping city values:")
print(df["shipping_city"].value_counts())

Shipping city values:
shipping_city
London       113
Kitchener    109
Ottawa        99
Toronto       94
Waterloo      82
 toronto       1
WATERLOO       1
kitchener      1
Name: count, dtype: int64


Summarize the identified problems

In [10]:
missing_prices = df["price"].isnull().sum()
missing_products = df["product"].isnull().sum()
missing_coupons = df["coupon_code"].isnull().sum()

print("Missing prices:", missing_prices)
print("Missing products:", missing_products)
print("Missing coupon codes:", missing_coupons)

Missing prices: 3
Missing products: 2
Missing coupon codes: 130


## Step 7 - Cleaning Rules

The `Transaction.clean()` method is used to clean the data. Missing prices are replaced with the average price, missing product names are changed to `Unknown`, and missing coupon codes are changed to `NO_COUPON`. Shipping city names are also corrected to have the same format. The results before and after cleaning are compared to make sure the data was cleaned correctly.

Record the BEFORE values

In [11]:
before_missing_price = df["price"].isnull().sum()
before_missing_product = df["product"].isnull().sum()
before_missing_coupon = df["coupon_code"].isnull().sum()
before_unique_cities = df["shipping_city"].nunique()

print("Before Cleaning")
print("Missing prices:", before_missing_price)
print("Missing products:", before_missing_product)
print("Missing coupons:", before_missing_coupon)
print("Unique cities:", before_unique_cities)

Before Cleaning
Missing prices: 3
Missing products: 2
Missing coupons: 130
Unique cities: 8


In [12]:
mean_price = df["price"].mean()

In [14]:
cleaned_transactions = []

for _, row in df.iterrows():

    transaction = Transaction(
        transaction_id=row["transaction_id"],
        date=row["date"],
        customer_id=row["customer_id"],
        product=row["product"],
        price=row["price"],
        quantity=row["quantity"],
        coupon_code=row["coupon_code"],
        shipping_city=row["shipping_city"]
    )

    transaction.clean(mean_price)

    cleaned_transactions.append({
        "transaction_id": transaction.transaction_id,
        "date": transaction.date,
        "customer_id": transaction.customer_id,
        "product": transaction.product,
        "price": transaction.price,
        "quantity": transaction.quantity,
        "coupon_code": transaction.coupon_code,
        "shipping_city": transaction.shipping_city
    })

In [15]:
clean_df = pd.DataFrame(cleaned_transactions)

clean_df.head(3)

,transaction_id,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,1,2026-06-10,CUST5572,Smartwatch,367.57,2,NO_COUPON,London
1,2,2026-06-10,CUST8096,Laptop,433.53,1,SAVE10,Waterloo
2,3,2026-09-14,CUST6154,Tablet,1001.78,2,SAVE10,London


Record the AFTER values

In [16]:
print("After Cleaning")
print("Missing prices:", clean_df["price"].isnull().sum())
print("Missing products:", clean_df["product"].isnull().sum())
print("Missing coupons:", clean_df["coupon_code"].isnull().sum())
print("Unique cities:", clean_df["shipping_city"].nunique())

After Cleaning
Missing prices: 0
Missing products: 0
Missing coupons: 0
Unique cities: 5


## Step 8 - Transformations

The `coupon_code` values are converted into numeric discount percentages. For example, `SAVE10` is converted to `10`, `SAVE20` to `20`, `SAVE30` to `30`, and `NO_COUPON` to `0`.

The `subtotal` is calculated by multiplying the product price by the quantity. The `discount_amount` is calculated using the discount percentage, and the `final_amount` is calculated by subtracting the discount amount from the subtotal.

In [17]:
discount_map = {
    "SAVE10": 10,
    "SAVE20": 20,
    "SAVE30": 30,
    "NO_COUPON": 0
}

clean_df["discount_percent"] = clean_df["coupon_code"].map(discount_map)

clean_df[["coupon_code", "discount_percent"]].head(10)

,coupon_code,discount_percent
0,NO_COUPON,0
1,SAVE10,10
2,SAVE10,10
3,SAVE30,30
4,NO_COUPON,0
5,NO_COUPON,0
6,NO_COUPON,0
7,SAVE10,10
8,SAVE20,20
9,SAVE30,30


In [18]:
clean_df["subtotal"] = (
    clean_df["price"] * clean_df["quantity"]
).round(2)

clean_df["discount_amount"] = (
    clean_df["subtotal"] * clean_df["discount_percent"] / 100
).round(2)

clean_df["final_amount"] = (
    clean_df["subtotal"] - clean_df["discount_amount"]
).round(2)

clean_df[
    [
        "price",
        "quantity",
        "coupon_code",
        "discount_percent",
        "subtotal",
        "discount_amount",
        "final_amount"
    ]
].head()

,price,quantity,coupon_code,discount_percent,subtotal,discount_amount,final_amount
0,367.57,2,NO_COUPON,0,735.14,0.00,735.14
1,433.53,1,SAVE10,10,433.53,43.35,390.18
2,1001.78,2,SAVE10,10,2003.56,200.36,1803.20
3,1029.17,4,SAVE30,30,4116.68,1235.00,2881.68
4,21.44,5,NO_COUPON,0,107.20,0.00,107.20


## Step 9 - Feature Engineering

A new feature called `days_since_purchase` is created from the transaction date. It represents the number of days between each purchase and the most recent transaction date in the dataset.

In [19]:
clean_df["date"] = pd.to_datetime(clean_df["date"])

latest_date = clean_df["date"].max()

clean_df["days_since_purchase"] = (
    latest_date - clean_df["date"]
).dt.days

clean_df[["date", "days_since_purchase"]].head(10)

,date,days_since_purchase
0,2026-06-10,102
1,2026-06-10,102
2,2026-09-14,6
3,2026-06-10,102
4,2026-09-10,10
5,2026-08-28,23
6,2025-10-01,354
7,2026-03-16,188
8,2026-08-04,47
9,2026-03-07,197


In [20]:
print("Latest transaction date:", latest_date)
print("Minimum days since purchase:", clean_df["days_since_purchase"].min())
print("Maximum days since purchase:", clean_df["days_since_purchase"].max())

Latest transaction date: 2026-09-20 00:00:00
Minimum days since purchase: 0
Maximum days since purchase: 363


## Step 10 - Mini-Aggregation

The cleaned transaction data is grouped by shipping city to calculate the total revenue generated from each city. The `final_amount` is used so that coupon discounts are included in the revenue calculation.

In [21]:
revenue_by_city = (
    clean_df.groupby("shipping_city")["final_amount"]
    .sum()
    .round(2)
    .sort_values(ascending=False)
)

revenue_by_city

shipping_city
London       316483.42
Toronto      280344.29
Ottawa       271743.14
Kitchener    256928.42
Waterloo     227064.70
Name: final_amount, dtype: float64

## Secondary Metadata Source

A Statistics Canada open-data file is used as the secondary metadata source. The dataset provides Canadian geographic and population information that can be used to supplement the shipping-city information in the e-commerce dataset.

In [22]:
column_names = [
    "geographic_name",
    "geographic_area_type",
    "province",
    "population_2021",
    "population_2016",
    "population_change_percent",
    "total_dwellings_2021",
    "total_dwellings_2016",
    "dwellings_change_percent",
    "occupied_dwellings_2021",
    "occupied_dwellings_2016",
    "occupied_dwellings_change_percent",
    "land_area_km2",
    "population_density_km2",
    "national_population_rank",
    "province_population_rank"
]

city_data = pd.read_csv(
    "data/9810000201-noSymbol.csv",
    skiprows=10,
    names=column_names,
    thousands=","
)


city_data.head()

,geographic_name,geographic_area_type,province,population_2021,population_2016,population_change_percent,total_dwellings_2021,total_dwellings_2016,dwellings_change_percent,occupied_dwellings_2021,occupied_dwellings_2016,occupied_dwellings_change_percent,land_area_km2,population_density_km2,national_population_rank,province_population_rank
0,Canada 2,Country,...,"36,991,981","35,151,728",5.2,"16,284,235","15,412,443",5.7,"14,978,941","14,072,079",6.4,8788702.80,4.2,...,...
1,Admirals Beach,T,N.L.,97,135,-28.1,76,80,-5.0,48,62,-22.6,24.20,4.0,"4,267",325
2,Aquaforte,T,N.L.,74,80,-7.5,63,71,-11.3,43,41,4.9,6.88,10.7,"4,387",339
3,Arnold's Cove,T,N.L.,964,949,1.6,547,537,1.9,434,392,10.7,5.25,183.8,"2,155",79
4,Avondale,T,N.L.,584,641,-8.9,346,373,-7.2,286,294,-2.7,29.69,19.7,"2,741",130


In [23]:
print("Dataset shape:", city_data.shape)

city_data[
    [
        "geographic_name",
        "province",
        "population_2021",
        "land_area_km2",
        "population_density_km2"
    ]
].head()

Dataset shape: (5180, 16)


,geographic_name,province,population_2021,land_area_km2,population_density_km2
0,Canada 2,...,"36,991,981",8788702.80,4.2
1,Admirals Beach,N.L.,97,24.20,4.0
2,Aquaforte,N.L.,74,6.88,10.7
3,Arnold's Cove,N.L.,964,5.25,183.8
4,Avondale,N.L.,584,29.69,19.7


### Extracting Relevant City Metadata

Only the Ontario cities that appear in the e-commerce transaction dataset are selected from the Statistics Canada data. Population, land area, and population density are retained as useful metadata for each shipping city.

In [24]:
selected_cities = [
    "Toronto",
    "Waterloo",
    "Kitchener",
    "Ottawa",
    "London"
]

city_metadata = city_data[
    (city_data["geographic_name"].isin(selected_cities)) &
    (city_data["province"] == "Ont.")
][
    [
        "geographic_name",
        "population_2021",
        "land_area_km2",
        "population_density_km2"
    ]
].copy()

city_metadata

,geographic_name,population_2021,land_area_km2,population_density_km2
2393,London,"422,324",420.50,"1,004.3"
2446,Ottawa,"1,017,449",2788.20,364.9
2665,Toronto,"2,794,356",631.10,"4,427.8"
2667,Kitchener,"256,885",136.81,"1,877.6"
2669,Waterloo,"121,436",64.06,"1,895.8"


In [25]:
city_metadata = city_metadata.rename(
    columns={"geographic_name": "city"}
)

city_metadata

,city,population_2021,land_area_km2,population_density_km2
2393,London,"422,324",420.50,"1,004.3"
2446,Ottawa,"1,017,449",2788.20,364.9
2665,Toronto,"2,794,356",631.10,"4,427.8"
2667,Kitchener,"256,885",136.81,"1,877.6"
2669,Waterloo,"121,436",64.06,"1,895.8"


In [26]:
city_metadata.to_csv(
    "data/city_metadata.csv",
    index=False
)

print("City metadata saved successfully.")

City metadata saved successfully.


### Merging the Two Data Sources

The cleaned e-commerce transactions are merged with the Statistics Canada city metadata using the shipping city as the common field. A left join is used so that all transaction records are kept while population, land area, and population density are added from the secondary dataset.

In [27]:
clean_df = clean_df.merge(
    city_metadata,
    left_on="shipping_city",
    right_on="city",
    how="left"
)

clean_df.head()

,transaction_id,date,customer_id,product,price,quantity,coupon_code,shipping_city,discount_percent,subtotal,discount_amount,final_amount,days_since_purchase,city,population_2021,land_area_km2,population_density_km2
0,1,2026-06-10,CUST5572,Smartwatch,367.57,2,NO_COUPON,London,0,735.14,0.00,735.14,102,London,"422,324",420.50,"1,004.3"
1,2,2026-06-10,CUST8096,Laptop,433.53,1,SAVE10,Waterloo,10,433.53,43.35,390.18,102,Waterloo,"121,436",64.06,"1,895.8"
2,3,2026-09-14,CUST6154,Tablet,1001.78,2,SAVE10,London,10,2003.56,200.36,1803.20,6,London,"422,324",420.50,"1,004.3"
3,4,2026-06-10,CUST5695,Smartphone,1029.17,4,SAVE30,Toronto,30,4116.68,1235.00,2881.68,102,Toronto,"2,794,356",631.10,"4,427.8"
4,5,2026-09-10,CUST8276,Mouse,21.44,5,NO_COUPON,London,0,107.20,0.00,107.20,10,London,"422,324",420.50,"1,004.3"


## Step 11 - Serialization Checkpoint

The cleaned and transformed transaction data is saved in both CSV and JSON formats. Saving the processed dataset creates a reusable checkpoint that can be loaded later without repeating the cleaning and transformation steps.

In [28]:
clean_df.to_csv(
    "data/cleaned_ecommerce_transactions.csv",
    index=False
)

clean_df.to_json(
    "data/cleaned_ecommerce_transactions.json",
    orient="records",
    indent=4,
    date_format="iso"
)

print("Cleaned data saved successfully in CSV and JSON formats.")

Cleaned data saved successfully in CSV and JSON formats.


In [29]:
csv_check = pd.read_csv("data/cleaned_ecommerce_transactions.csv")
json_check = pd.read_json("data/cleaned_ecommerce_transactions.json")

print("CSV records:", len(csv_check))
print("JSON records:", len(json_check))

CSV records: 500
JSON records: 500


## Step 12 - Soft Interview Reflection

Functions helped me organize the code and avoid repeating the same steps. I used the `clean()` method to apply the same cleaning rules to every transaction. I also used the `total()` method to calculate the total price of a transaction. Using these methods made the code easier to understand and reuse. If I need to change a cleaning rule later, I can update it in one place instead of changing the code for every transaction.

## Data Dictionary

The following data dictionary describes the fields used in the project. It includes fields from the primary e-commerce transaction dataset, fields created during data processing, and selected metadata obtained from Statistics Canada.

| Field | Type | Description | Source |
|---|---|---|---|
| transaction_id | Integer | Unique identifier for each transaction | Synthetic e-commerce dataset |
| date | Date | Date when the transaction occurred | Synthetic e-commerce dataset |
| customer_id | String | Unique identifier for the customer | Synthetic e-commerce dataset |
| product | String | Name of the purchased product | Synthetic e-commerce dataset |
| price | Float | Price of one unit of the product | Synthetic e-commerce dataset |
| quantity | Integer | Number of units purchased | Synthetic e-commerce dataset |
| coupon_code | String | Promotional code used for the transaction; missing values are changed to `NO_COUPON` | Synthetic e-commerce dataset / Cleaning |
| shipping_city | String | City where the order is shipped | Synthetic e-commerce dataset |
| discount_percent | Integer | Numeric discount percentage derived from the coupon code | Derived |
| subtotal | Float | Price multiplied by quantity before discount | Derived |
| discount_amount | Float | Amount deducted from the subtotal based on the discount percentage | Derived |
| final_amount | Float | Final transaction amount after discount | Derived |
| days_since_purchase | Integer | Number of days between the purchase and the latest transaction date in the dataset | Derived |
| city | String | Name of the Canadian city | Statistics Canada |
| population_2021 | Integer | Population of the city in the 2021 Census | Statistics Canada |
| land_area_km2 | Float | Land area of the city in square kilometres | Statistics Canada |
| population_density_km2 | Float | Population per square kilometre in 2021 | Statistics Canada |

### Cleaning and Derived Field Notes

Missing product names are replaced with `Unknown`, missing coupon codes are replaced with `NO_COUPON`, and inconsistent shipping-city names are standardized. Missing prices are replaced with the mean price of the available transactions. The discount and transaction amount fields are calculated from the cleaned transaction data.

## Data Dictionary

The following table describes the fields used in this project. It includes fields from the synthetic e-commerce dataset, fields created during data cleaning and transformation, and fields selected from the Statistics Canada dataset.

| Field | Type | Description | Source / How Created |
|---|---|---|---|
| transaction_id | Integer | Unique ID for each transaction | Synthetic e-commerce dataset |
| date | Date | Date when the transaction occurred | Synthetic e-commerce dataset |
| customer_id | String | Unique ID for each customer | Synthetic e-commerce dataset |
| product | String | Name of the purchased product | Synthetic data; missing values replaced with `Unknown` |
| price | Float | Price of one unit of the product | Synthetic data; missing values replaced with the mean price |
| quantity | Integer | Number of units purchased | Synthetic e-commerce dataset |
| coupon_code | String | Coupon used for the transaction | Synthetic data; missing values replaced with `NO_COUPON` |
| shipping_city | String | City where the order is shipped | Synthetic data; city names cleaned to a consistent format |
| discount_percent | Integer | Numeric discount percentage | Derived from `coupon_code` |
| subtotal | Float | Transaction amount before discount | Derived: `price × quantity` |
| discount_amount | Float | Amount discounted from the subtotal | Derived: `subtotal × discount_percent / 100` |
| final_amount | Float | Final amount after applying the discount | Derived: `subtotal - discount_amount` |
| days_since_purchase | Integer | Number of days between the purchase and the latest transaction date | Derived from `date` and the latest transaction date |
| geographic_name | String | Name of the city | Statistics Canada |
| population_2021 | Integer | Population of the city in 2021 | Statistics Canada |
| land_area_km2 | Float | Land area of the city in square kilometres | Statistics Canada |
| population_density_km2 | Float | Number of people per square kilometre | Statistics Canada |